# Launch Nemotron Voice Agent on Brev

This notebook launches the Nemotron Voice Agent on a Brev GPU instance using the repository's Docker Compose configuration. It prepares the `.env` file, starts the required services, and verifies that the web application and WebRTC networking are ready.

The Brev Launchable provides this repository on the instance. The notebook auto-detects `REPO_ROOT` from the current directory, parent directories, and common Brev workspace locations; update it in the settings cell only if your repository is in a custom location.


## What Will Be Deployed

This launch path uses the Docker Compose recipe profile `cascaded/generic/workstation` plus the `turn` profile. Together, they deploy:

- The Nemotron Voice Agent web application on port `7860`.
- Local NIM services for ASR, LLM, and TTS through the workstation recipe.
- A coturn service for WebRTC connectivity, using port `3478` and relay range `49160-49200`.

For Brev Secure Links, the application is served over HTTP by setting `PIPELINE_TLS=false`. The repository default remains TLS-enabled when this setting is not applied.

The diagram below shows the high-level voice-agent runtime used by this deployment.

![Nemotron Voice Agent architecture](../docs/images/arch.png)


## 1. Prerequisites and User Settings

Before running the notebook, confirm the Brev instance has:

- A workstation-class NVIDIA GPU setup for the `workstation` hardware profile, paired with the `cascaded/generic` example profile. This profile uses local NIM services for ASR, TTS, and LLM; the default compose configuration expects two GPUs.
- Docker Compose available and configured for NVIDIA GPUs.
- An NVIDIA API key with access to NGC containers and NVIDIA-hosted model services. You can create one from [NVIDIA API Keys](https://build.nvidia.com/settings/api-keys).
- Brev networking configured for TURN. Since the `turn` profile deploys coturn on the same instance, expose port `3478` and range `49160-49200` in the Brev UI.

Usually you only need to set `NVIDIA_API_KEY`. Update `REPO_ROOT_OVERRIDE` only if automatic repo discovery does not find your checkout. Set `RESET_VOLUMES=True` only when you want to delete Docker/NIM caches and start fully fresh.


In [ ]:
from __future__ import annotations

import getpass
import json
import os
import secrets
import subprocess
import time
import urllib.request
from pathlib import Path

REPO_ROOT_OVERRIDE = os.environ.get("REPO_ROOT", "")

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
TURN_USERNAME = os.environ.get("TURN_USERNAME", "")
TURN_PASSWORD = os.environ.get("TURN_PASSWORD", "")
TURN_URL = os.environ.get("TURN_URL", "")
PIPELINE_APP_PORT = os.environ.get("PIPELINE_APP_PORT", "7860")

PROFILES = ["cascaded/generic/workstation", "turn"]
AUTO_DETECT_TURN_URL = True
RESET_VOLUMES = False
PRUNE_BUILD_CACHE = True
CONFIGURE_DOCKER_DATA_ROOT = True
DOCKER_DATA_DIR_NAME = "docker"

# Brev Secure Links proxy HTTP to the app, so use no TLS for this launch path.
PIPELINE_TLS = "false"


def find_repo_root() -> Path:
    """Find the repository root from common Brev/Jupyter launch locations."""
    if REPO_ROOT_OVERRIDE:
        return Path(REPO_ROOT_OVERRIDE).expanduser().resolve()

    candidates: list[Path] = []
    search_roots = [Path.cwd(), Path.home(), Path("/home"), Path("/workspace"), Path("/opt/dlami/nvme")]
    for root in search_roots:
        if not root.exists():
            continue
        candidates.extend([root, *root.parents])
        if root != Path("/"):
            for pattern in ("*", "*/*", "*/*/*", "*/*/*/*"):
                candidates.extend(root.glob(pattern))

    for candidate in dict.fromkeys(candidates):
        try:
            has_markers = (candidate / "docker-compose.yml").exists() and (
                candidate / "examples_registry.yaml"
            ).exists()
        except OSError:
            has_markers = False
        if has_markers:
            return candidate.resolve()

    raise RuntimeError(
        "Repository root not found. Set REPO_ROOT_OVERRIDE to the checkout path, "
        "for example /home/ubuntu/path/to/nemotron-voice-agent."
    )


REPO_ROOT = find_repo_root()
ENV_PATH = REPO_ROOT / ".env"
ENV_EXAMPLE_PATH = REPO_ROOT / ".env.example"
DOCKER = ["docker"]
COMPOSE = [*DOCKER, "compose"]
COMPOSE_PROGRESS = [*DOCKER, "compose", "--progress", "plain"]
for profile in PROFILES:
    COMPOSE.extend(["--profile", profile])
    COMPOSE_PROGRESS.extend(["--profile", profile])

print(f"Repo: {REPO_ROOT}")
print(f"Profiles: {', '.join(PROFILES)}")

## 2. Notebook Utilities

These small utilities keep command output readable, redact secrets, and perform simple HTTP checks used later in the notebook.


In [ ]:
def redact(text: str) -> str:
    """Redact secrets before printing command output."""
    for value in (NVIDIA_API_KEY, TURN_PASSWORD):
        if value:
            text = text.replace(value, "***REDACTED***")
    return text


def run(
    cmd: list[str],
    *,
    check: bool = True,
    input_text: str | None = None,
    stream: bool = False,
):
    """Run a command from the repository root and print redacted output."""
    print("$ " + redact(" ".join(cmd)))
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            stdin=subprocess.PIPE if input_text is not None else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if input_text is not None and proc.stdin:
            proc.stdin.write(input_text)
            proc.stdin.close()
        output: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            output.append(line)
            stripped = line.rstrip()
            if stripped.endswith("Pulling fs layer 0B"):
                continue
            print(redact(stripped))
        proc.wait()
        stdout = "".join(output)
        if check and proc.returncode != 0:
            raise subprocess.CalledProcessError(proc.returncode, cmd, stdout, "")
        return subprocess.CompletedProcess(cmd, proc.returncode, stdout, "")

    proc = subprocess.run(cmd, cwd=REPO_ROOT, input=input_text, text=True, capture_output=True)
    if proc.stdout:
        print(redact(proc.stdout.rstrip()))
    if proc.stderr:
        print(redact(proc.stderr.rstrip()))
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, proc.stdout, proc.stderr)
    return proc


def fetch_text(url: str, timeout: int = 10) -> str:
    """Fetch a URL and decode the response as text."""
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return response.read().decode("utf-8", errors="replace")


def detect_public_ip() -> str:
    """Detect the Brev instance public IP for TURN."""
    return urllib.request.urlopen("https://ifconfig.me", timeout=10).read().decode().strip()

## 3. Configure Docker Storage

Brev machines can expose the largest disk at different paths, so do not assume `/data`. The repo can live anywhere; Docker images and volumes should be on the largest writable data mount.

This cell discovers that mount and, when `CONFIGURE_DOCKER_DATA_ROOT=True`, configures Docker to use it before large NIM images are pulled. It does not migrate existing Docker images; run this early on a fresh instance.

In [ ]:
def largest_data_mount() -> Path:
    """Return the largest writable non-system mount by available space."""
    ignored_fstypes = {"tmpfs", "devtmpfs", "overlay", "squashfs"}
    ignored_prefixes = ("/boot", "/dev", "/proc", "/run", "/snap", "/sys")
    candidates: list[tuple[int, Path]] = []

    for line in Path("/proc/mounts").read_text().splitlines():
        parts = line.split()
        if len(parts) < 4:
            continue
        mount = Path(parts[1].replace("\\040", " "))
        fstype = parts[2]
        options = parts[3].split(",")
        if "ro" in options or fstype in ignored_fstypes or str(mount).startswith(ignored_prefixes):
            continue
        try:
            stats = os.statvfs(mount)
        except OSError:
            continue
        candidates.append((stats.f_bavail * stats.f_frsize, mount))

    if not candidates:
        return Path("/")
    return max(candidates)[1]


def current_docker_root() -> Path | None:
    """Return Docker's configured data root when Docker is available."""
    try:
        proc = run([*DOCKER, "info", "--format", "{{.DockerRootDir}}"], check=False)
    except FileNotFoundError:
        print("Docker is not installed or not on PATH yet.")
        return None
    root = proc.stdout.strip()
    return Path(root) if proc.returncode == 0 and root else None


def docker_has_running_containers() -> bool:
    """Return whether Docker currently has running containers."""
    proc = run([*DOCKER, "ps", "-q"], check=False)
    return bool(proc.stdout.strip())


def write_docker_daemon_config(data_root: Path) -> None:
    """Write Docker daemon configuration with the requested data root."""
    proc = subprocess.run(
        ["sudo", "sh", "-c", "test -f /etc/docker/daemon.json && cat /etc/docker/daemon.json || printf '{}'"],
        text=True,
        capture_output=True,
        check=True,
    )
    config = json.loads(proc.stdout or "{}")
    config["data-root"] = str(data_root)
    tmp_path = Path("/tmp/nemotron-voice-agent-daemon.json")
    tmp_path.write_text(json.dumps(config, indent=2) + "\n")
    run(["sudo", "mkdir", "-p", str(data_root), "/etc/docker"])
    run(["sudo", "cp", str(tmp_path), "/etc/docker/daemon.json"])


def configure_docker_data_root(current_root: Path, target_root: Path) -> Path:
    """Move Docker's data root to the largest data mount when configured."""
    if Path("/") == LARGEST_DATA_MOUNT or current_root.resolve().is_relative_to(LARGEST_DATA_MOUNT):
        return current_root
    if not CONFIGURE_DOCKER_DATA_ROOT:
        print("Docker is not using the largest data mount; set CONFIGURE_DOCKER_DATA_ROOT=True to update it.")
        return current_root
    if docker_has_running_containers():
        raise RuntimeError("Docker has running containers. Stop them before changing Docker data-root.")

    print(f"Configuring Docker data-root: {target_root}")
    run(["sudo", "systemctl", "stop", "docker.service", "docker.socket"], check=False)
    write_docker_daemon_config(target_root)
    run(["sudo", "systemctl", "start", "docker.service"])
    verified_root = current_docker_root()
    if not verified_root or not verified_root.resolve().is_relative_to(LARGEST_DATA_MOUNT):
        raise RuntimeError(f"Docker data-root did not move to {LARGEST_DATA_MOUNT}.")
    return verified_root


LARGEST_DATA_MOUNT = largest_data_mount()
RECOMMENDED_DOCKER_DATA_ROOT = LARGEST_DATA_MOUNT / DOCKER_DATA_DIR_NAME
initial_docker_root = current_docker_root()
DOCKER_DATA_ROOT = (
    configure_docker_data_root(initial_docker_root, RECOMMENDED_DOCKER_DATA_ROOT)
    if initial_docker_root
    else RECOMMENDED_DOCKER_DATA_ROOT
)

print(f"Largest data mount: {LARGEST_DATA_MOUNT}")
print(f"Recommended Docker data-root: {RECOMMENDED_DOCKER_DATA_ROOT}")
print(f"Current Docker data-root: {DOCKER_DATA_ROOT}")

## 4. Preflight

Confirm Docker, Docker Compose, selected storage, and GPU visibility before deployment. The important check is that `DockerRootDir` points at the discovered large data mount, for example `/opt/dlami/nvme/docker` on Brev images that expose the large NVMe disk there.

If Docker or Docker Compose is not installed, install Docker Engine using NVIDIA's preferred environment setup for your Brev image, then rerun the storage setup and this section. On Ubuntu-based images, the Docker convenience script is often the fastest path:

```bash
curl -fsSL https://get.docker.com | sh
sudo usermod -aG docker $USER
newgrp docker
```

If `nvidia-smi` is not available or Docker cannot access GPUs, fix the Brev instance GPU/runtime setup before continuing.


In [ ]:
run([*DOCKER, "ps"])
run([*DOCKER, "info", "--format", "DockerRootDir={{.DockerRootDir}} default={{.DefaultRuntime}}"], check=False)
run([*DOCKER, "--version"])
run([*DOCKER, "compose", "version"])
df_paths = list(dict.fromkeys([str(REPO_ROOT), str(DOCKER_DATA_ROOT), str(RECOMMENDED_DOCKER_DATA_ROOT)]))
run(["df", "-h", *df_paths], check=False)
run(["nvidia-smi"], check=False)

## 5. Configure `.env`

This step writes the runtime settings into `.env` because Docker Compose and the application containers read configuration from that file. Exporting `NVIDIA_API_KEY` in the notebook kernel is not enough for every container path.

`TURN_URL` is set to the instance public IP, not the Brev Secure Link hostname. `PIPELINE_TLS=false` makes the app serve HTTP for Brev Secure Link compatibility.


In [ ]:
def read_env(path: Path) -> list[str]:
    """Read an env file into individual lines."""
    return path.read_text().splitlines() if path.exists() else []


def get_env(lines: list[str], key: str) -> str:
    """Return an env value from parsed lines."""
    for line in lines:
        stripped = line.strip()
        if stripped.startswith(f"{key}="):
            return stripped.split("=", 1)[1]
    return ""


def set_env(lines: list[str], key: str, value: str) -> list[str]:
    """Set or append an env value in parsed lines."""
    replacement = f"{key}={value}"
    for index, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith(f"{key}=") or stripped.startswith(f"# {key}=") or stripped.startswith(f"#{key}="):
            lines[index] = replacement
            return lines
    lines.append(replacement)
    return lines


if not ENV_PATH.exists():
    ENV_PATH.write_text(ENV_EXAMPLE_PATH.read_text())

lines = read_env(ENV_PATH)
NVIDIA_API_KEY = NVIDIA_API_KEY or get_env(lines, "NVIDIA_API_KEY")
TURN_USERNAME = TURN_USERNAME or get_env(lines, "TURN_USERNAME") or "admin"
TURN_PASSWORD = TURN_PASSWORD or get_env(lines, "TURN_PASSWORD")
TURN_URL = TURN_URL or get_env(lines, "TURN_URL")

if not NVIDIA_API_KEY or NVIDIA_API_KEY == "nvapi-...":
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise RuntimeError("NVIDIA_API_KEY is required.")

if not TURN_PASSWORD or TURN_PASSWORD == "admin":
    TURN_PASSWORD = secrets.token_urlsafe(32)
    print("Generated a random TURN_PASSWORD; it has been written to .env.")

if AUTO_DETECT_TURN_URL and (not TURN_URL or "brevlab.com" in TURN_URL):
    public_ip = detect_public_ip()
    TURN_URL = f"turn:{public_ip}:3478"
    print(f"Using TURN_URL={TURN_URL}")

values = {
    "NVIDIA_API_KEY": NVIDIA_API_KEY,
    "TURN_USERNAME": TURN_USERNAME,
    "TURN_PASSWORD": TURN_PASSWORD,
    "TURN_URL": TURN_URL,
    "PIPELINE_APP_PORT": PIPELINE_APP_PORT,
    "PIPELINE_TLS": PIPELINE_TLS,
}
for key, value in values.items():
    lines = set_env(lines, key, value)
ENV_PATH.write_text("\n".join(lines).rstrip() + "\n")
os.environ.update(values)
print(f"Updated {ENV_PATH}")

## 6. Deploy

Safe to rerun. Containers are recreated; model-cache volumes are preserved unless `RESET_VOLUMES=True`.


In [ ]:
run([*DOCKER, "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"], input_text=NVIDIA_API_KEY + "\n")
run([*COMPOSE, "config", "--services"])

down_cmd = [*COMPOSE, "down", "--remove-orphans"]
if RESET_VOLUMES:
    down_cmd.append("-v")
run(down_cmd, check=False)

if PRUNE_BUILD_CACHE:
    run([*DOCKER, "builder", "prune", "-f"], check=False)

run([*COMPOSE_PROGRESS, "up", "-d", "--build", "--remove-orphans"], stream=True)
run([*COMPOSE, "ps"])
run([*DOCKER, "system", "df"], check=False)

## 7. Verify

When `PIPELINE_TLS=false`, local health uses HTTP and Brev Secure Link can proxy to port `7860`.


In [ ]:
scheme = "http" if PIPELINE_TLS.lower() == "false" else "https"
health_url = f"{scheme}://localhost:{PIPELINE_APP_PORT}/health"
ice_url = f"{scheme}://localhost:{PIPELINE_APP_PORT}/api/ice-servers"

for attempt in range(1, 31):
    try:
        fetch_text(health_url, timeout=5)
        print(f"App healthy: {health_url}")
        break
    except Exception as exc:
        if attempt == 30:
            raise
        print(f"Waiting for app ({attempt}/30): {exc}")
        time.sleep(10)

try:
    ice = json.loads(fetch_text(ice_url, timeout=10))
    print(json.dumps(ice, indent=2))
except Exception as exc:
    print(f"ICE check failed: {exc}")

print(f"Open app using the Brev Secure Link for port {PIPELINE_APP_PORT}.")
print("TURN uses TURN_URL and needs UDP 3478 plus UDP 49160-49200 reachable.")

## 8. Exposing the Public Endpoint Using Brev UI

Use the Brev console to expose the TURN ports and create a Secure Link for the web app. The Secure Link is only for HTTP access to the app on port `7860`; TURN still needs the instance public IP and raw UDP/TCP ports.

### Expose TURN ports

Start from the Brev **Using Ports** section, where SSH port `22` is present by default.

![Brev ports before exposing TURN](images/brev-ports-before.png)

Expose port `3478` and range `49160-49200` with access allowed from any IP. These are used by coturn for WebRTC connectivity.

![Brev TURN ports exposed](images/brev-turn-ports.png)

### Create the app Secure Link

Open **Using Secure Links** and click **Share a Service**.

![Brev Secure Links before adding the app service](images/brev-secure-link-created.png)

Create a Secure Link for port `7860` and name it `nemotron-voice-agent`.

![Brev share service dialog for port 7860](images/brev-share-service-dialog.png)

Do not use the Secure Link hostname as `TURN_URL`; the notebook auto-detects the instance public IP for TURN.
